# 💼 NovaSuite — Pricing & Retention Strategy Analysis

---

## Section Overview

This section evaluates **pricing effectiveness** and identifies **retention opportunities** using insights from the previous Revenue and Churn analyses.

| # | Analysis | Business Question |
|---|----------|-------------------|
| 1 | Basic Plan Pricing | Is the Basic plan underpriced or over-discounted? |
| 2 | Pro Plan Pricing | Would a higher Pro price be justified by customer value? |
| 3 | Discount Leakage | Which segments receive the highest discounts? |
| 4 | Usage-Based Tier Opportunity | Would a usage-based model better serve Small customers? |
| 5 | Revenue-to-User Efficiency | Which plans are over-provisioned or underutilized? |
| 6 | Retention Opportunity Analysis | Which segments create the biggest retention risk? |
| 7 | High-Value At-Risk Customers | Which customers should be prioritized for retention? |
| 8 | Summary of Findings | Data-driven pricing and retention observations |


---

In [ ]:
import pandas as pd
import numpy as np

# --- Load raw tables ---
customers = pd.read_csv('customers.csv')
revenue   = pd.read_csv('revenue.csv')
usage     = pd.read_csv('usage.csv')
churn     = pd.read_csv('churn.csv')
plans     = pd.read_csv('plans.csv')

# --- Build master ---
# customers has plan_id; revenue also has plan_id — use suffix to avoid collision
master = customers.merge(revenue, on='customer_id', how='left', suffixes=('_cust', '_rev'))
master = master.merge(usage, on=['customer_id', 'month'], how='left')
master = master.merge(plans, left_on='plan_id_cust', right_on='plan_id', how='left')

print(f'master shape: {master.shape}')
print(f'Columns: {list(master.columns)}')

---

## 1. Basic Plan Pricing Analysis

**Business Question:** Is the Basic plan underpriced or heavily discounted?

The Basic plan has a list price of **\$29/month**. We compare this against the average realized (subscription) revenue to measure discount erosion and revenue leakage.

In [ ]:
# ── Filter to Basic plan rows ──────────────────────────────────────────────
basic = master[master['plan_name'] == 'Basic'].copy()

# ── Key metrics ───────────────────────────────────────────────────────────
list_price        = 29.00
avg_realized      = round(basic['subscription_revenue'].mean(), 2)
avg_discount      = round(basic['discount'].mean(), 2)
total_discount    = round(basic['discount'].sum(), 2)
total_base_rev    = round(basic['base_revenue'].sum(), 2)
pct_leakage       = round(total_discount / total_base_rev * 100, 2)
revenue_gap       = round(list_price - avg_realized, 2)
num_basic_customers = basic['customer_id'].nunique()

# ── Summary table ─────────────────────────────────────────────────────────
basic_summary = pd.DataFrame({
    'Metric': [
        'List Price',
        'Avg Realized Revenue',
        'Revenue Gap (List − Realized)',
        'Avg Monthly Discount per Row',
        'Total Discount Amount (All Time)',
        'Total Base Revenue (All Time)',
        'Discount Leakage %',
        'Number of Basic Customers'
    ],
    'Value': [
        f'${list_price}',
        f'${avg_realized}',
        f'${revenue_gap}',
        f'${avg_discount}',
        f'${total_discount:,.2f}',
        f'${total_base_rev:,.2f}',
        f'{pct_leakage}%',
        str(num_basic_customers)
    ]
})

print('Basic Plan Pricing Summary')
print('=' * 42)
print(basic_summary.to_string(index=False))

In [ ]:
print('Business Interpretation')
print('=' * 60)
print(f'''
List Price          : ${list_price:.2f}/month
Avg Realized Revenue: ${avg_realized:.2f}/month
Revenue Gap         : ${revenue_gap:.2f} per customer per month

The Basic plan has a discount leakage of {pct_leakage}% — meaning NovaSuite
collects only ${avg_realized} of every ${list_price} billed on this plan.

Over all time, ${total_discount:,.0f} in discounts have been applied against
${total_base_rev:,.0f} in base revenue across {num_basic_customers} Basic customers.

Annualized, this represents roughly ${total_discount / 39 * 12:,.0f} in forgone
revenue per year from the Basic plan alone.

The current list price of $29 is already the lowest tier — heavy discounting
here signals either aggressive sales incentives, poor price enforcement,
or a structural need to revisit discount policy for entry-level plans.
''')

---

## 2. Pro Plan Pricing Analysis

**Business Question:** Would a higher Pro price be justified based on customer value?

The Pro plan (\$79/month, up to 20 users) is NovaSuite's most popular plan by customer count. We examine realized revenue versus list price and its contribution to total revenue.

In [ ]:
# ── Filter to Pro plan rows ────────────────────────────────────────────────
pro = master[master['plan_name'] == 'Pro'].copy()

# ── Key metrics ───────────────────────────────────────────────────────────
pro_list_price      = 79.00
pro_avg_realized    = round(pro['subscription_revenue'].mean(), 2)
pro_avg_discount    = round(pro['discount'].mean(), 2)
pro_total_sub_rev   = round(pro['subscription_revenue'].sum(), 2)
pro_total_discount  = round(pro['discount'].sum(), 2)
pro_total_base      = round(pro['base_revenue'].sum(), 2)
pro_leakage_pct     = round(pro_total_discount / pro_total_base * 100, 2)
pro_revenue_gap     = round(pro_list_price - pro_avg_realized, 2)
num_pro_customers   = pro['customer_id'].nunique()
total_all_rev       = round(master['subscription_revenue'].sum(), 2)
pro_rev_share       = round(pro_total_sub_rev / total_all_rev * 100, 2)

# ── Summary table ─────────────────────────────────────────────────────────
pro_summary = pd.DataFrame({
    'Metric': [
        'List Price',
        'Avg Realized Revenue',
        'Revenue Gap (List − Realized)',
        'Avg Monthly Discount per Row',
        'Discount Leakage %',
        'Total Subscription Revenue (All Time)',
        'Share of Total Revenue',
        'Number of Pro Customers'
    ],
    'Value': [
        f'${pro_list_price}',
        f'${pro_avg_realized}',
        f'${pro_revenue_gap}',
        f'${pro_avg_discount}',
        f'{pro_leakage_pct}%',
        f'${pro_total_sub_rev:,.2f}',
        f'{pro_rev_share}%',
        str(num_pro_customers)
    ]
})

print('Pro Plan Pricing Summary')
print('=' * 46)
print(pro_summary.to_string(index=False))

In [ ]:
print('Business Interpretation')
print('=' * 60)
print(f'''
The Pro plan contributes {pro_rev_share}% of total subscription revenue —
making it NovaSuite's single largest revenue driver by volume.

Despite a list price of ${pro_list_price}, the average realized revenue is
only ${pro_avg_realized} — a gap of ${pro_revenue_gap}/month per customer.

The Pro plan has a {pro_leakage_pct}% discount leakage rate, slightly higher
than Basic ({pct_leakage}%), meaning discounting is more aggressive at
this tier in absolute dollar terms (avg ${pro_avg_discount}/month off).

Given that Pro customers represent the majority of the customer base
and drive over a third of total revenue, even a modest price increase
combined with reduced discounting would have an outsized MRR impact.

The scale of this plan (177 customers) means each $1 increase in ARPA
translates to approximately $177/month or ~$2,124/year in incremental MRR.
''')

---

## 3. Discount Leakage Analysis

**Business Question:** Which segments receive the highest discounts?

Discount leakage erodes MRR without necessarily improving retention. We quantify discount amounts by plan and industry to identify where discount policy is weakest.

In [ ]:
# ── Discount leakage by plan ───────────────────────────────────────────────
disc_by_plan = master.groupby('plan_name')[['discount', 'base_revenue']].sum().reset_index()
disc_by_plan['avg_discount_per_row'] = (
    master.groupby('plan_name')['discount'].mean().values
)
disc_by_plan['discount_pct']         = (disc_by_plan['discount'] / disc_by_plan['base_revenue'] * 100).round(2)
disc_by_plan['discount']             = disc_by_plan['discount'].round(2)
disc_by_plan['base_revenue']         = disc_by_plan['base_revenue'].round(2)
disc_by_plan['avg_discount_per_row'] = disc_by_plan['avg_discount_per_row'].round(2)
disc_by_plan                         = disc_by_plan.sort_values('discount_pct', ascending=False)
disc_by_plan.columns = ['Plan', 'Total Discount ($)', 'Total Base Rev ($)',
                         'Avg Discount/Month ($)', 'Discount Leakage (%)']

print('Discount Leakage by Plan')
print('=' * 75)
print(disc_by_plan.to_string(index=False))

In [ ]:
# ── Discount leakage by industry ──────────────────────────────────────────
disc_by_ind = master.groupby('industry')[['discount', 'base_revenue']].sum().reset_index()
disc_by_ind['discount_pct'] = (disc_by_ind['discount'] / disc_by_ind['base_revenue'] * 100).round(2)
disc_by_ind['discount']     = disc_by_ind['discount'].round(2)
disc_by_ind['base_revenue'] = disc_by_ind['base_revenue'].round(2)
disc_by_ind                 = disc_by_ind.sort_values('discount_pct', ascending=False)
disc_by_ind.columns         = ['Industry', 'Total Discount ($)', 'Total Base Rev ($)',
                                 'Discount Leakage (%)']

print('Discount Leakage by Industry')
print('=' * 65)
print(disc_by_ind.to_string(index=False))

In [ ]:
# ── Total discount leakage summary ────────────────────────────────────────
total_disc_all   = round(master['discount'].sum(), 2)
total_base_all   = round(master['base_revenue'].sum(), 2)
overall_leakage  = round(total_disc_all / total_base_all * 100, 2)
months_in_data   = master['month'].nunique()
monthly_avg_disc = round(total_disc_all / months_in_data, 0)
annualized_disc  = round(monthly_avg_disc * 12, 0)

print('Overall Discount Leakage Summary')
print('=' * 60)
print(f'''
Total Discount Applied (All Time) : ${total_disc_all:,.2f}
Total Base Revenue (All Time)     : ${total_base_all:,.2f}
Overall Discount Leakage %        : {overall_leakage}%
Avg Monthly Discount Leakage      : ${monthly_avg_disc:,.0f}/month
Annualized Discount Leakage       : ${annualized_disc:,.0f}/year

Key Observations:
  - Enterprise plan has the highest leakage at 19.57% (avg $58.51 off/month)
  - Business plan follows at 16.39% (avg $24.42 off/month)
  - EdTech is the worst industry at 22.91% leakage — highest of all segments
  - Retail is second-worst at 17.56%, followed by Manufacturing at 15.64%
  - FinTech has the lowest industry leakage at 12.07%

Discount leakage is highest precisely where per-seat prices are highest
(Enterprise, Business), suggesting that sales teams are using heavy discounts
to close larger deals — but this strategy is not reflected in better retention.
''')

---

## 4. Usage-Based Tier Opportunity

**Business Question:** Would a usage-based tier better serve Small customers?

The proposed usage-based model: **\$20 base + \$8 per active user/month**.
We focus on Small customers on the Pro plan — the group most likely to be over-paying relative to actual usage.

In [ ]:
# ── Small customers on Pro plan ───────────────────────────────────────────
small_pro = master[
    (master['company_size'] == 'Small') &
    (master['plan_name'] == 'Pro')
].copy()

# ── Plan distribution for all Small customers ─────────────────────────────
small_all  = master[master['company_size'] == 'Small'].copy()
small_dist = small_all.groupby('plan_name')['customer_id'].nunique().reset_index()
small_dist.columns = ['Plan', 'Unique Customers']

print('Small Customer Plan Distribution')
print('=' * 35)
print(small_dist.to_string(index=False))
print()

In [ ]:
# ── Usage-based pricing simulation for Small + Pro ────────────────────────
small_pro['usage_based_rev'] = 20 + (8 * small_pro['active_users'])

avg_active_users   = round(small_pro['active_users'].mean(), 2)
avg_sessions       = round(small_pro['sessions'].mean(), 2)
avg_current_rev    = round(small_pro['subscription_revenue'].mean(), 2)
avg_ub_rev         = round(small_pro['usage_based_rev'].mean(), 2)
revenue_diff       = round(avg_current_rev - avg_ub_rev, 2)
pct_diff           = round(revenue_diff / avg_current_rev * 100, 2)
num_small_pro      = small_pro['customer_id'].nunique()

# Churn rate for Small+Pro vs Small+Basic
churned_ids        = set(churn['customer_id'])
seg_for_churn      = customers.merge(plans, on='plan_id').copy()
seg_for_churn['churned'] = seg_for_churn['customer_id'].isin(churned_ids).astype(int)

small_pro_seg      = seg_for_churn[(seg_for_churn['company_size'] == 'Small') & (seg_for_churn['plan_name'] == 'Pro')]
small_basic_seg    = seg_for_churn[(seg_for_churn['company_size'] == 'Small') & (seg_for_churn['plan_name'] == 'Basic')]
small_pro_churn    = round(small_pro_seg['churned'].mean() * 100, 1)
small_basic_churn  = round(small_basic_seg['churned'].mean() * 100, 1)

ub_summary = pd.DataFrame({
    'Metric': [
        'Small+Pro Customers',
        'Avg Active Users / Month',
        'Avg Sessions / Month',
        'Avg Current Realized Revenue',
        'Avg Revenue under Usage-Based ($20 + $8/user)',
        'Revenue Difference (Current − Usage-Based)',
        'Revenue Difference %',
        'Small+Pro Churn Rate',
        'Small+Basic Churn Rate (Benchmark)'
    ],
    'Value': [
        str(num_small_pro),
        str(avg_active_users),
        str(avg_sessions),
        f'${avg_current_rev}',
        f'${avg_ub_rev}',
        f'${revenue_diff}',
        f'{pct_diff}%',
        f'{small_pro_churn}%',
        f'{small_basic_churn}%'
    ]
})

print('Small + Pro Plan — Usage-Based Tier Simulation')
print('=' * 55)
print(ub_summary.to_string(index=False))

In [ ]:
print('Business Interpretation')
print('=' * 60)
print(f'''
There are {num_small_pro} Small customers on the Pro plan (list price $79/month).
Their average active user count is only {avg_active_users} — well below the 20-user
cap that Pro allows.

Under the proposed usage-based model ($20 + $8/active user):
  - Average revenue would be ${avg_ub_rev}/month vs ${avg_current_rev} currently
  - That is a ${revenue_diff} reduction ({pct_diff}% less per customer)

However, this segment has a {small_pro_churn}% churn rate — more than double
the {small_basic_churn}% rate seen in Small+Basic customers.

This suggests Small customers on Pro are paying for capacity they don't use,
which contributes to perceived poor value and eventual churn.

A usage-based tier would lower per-customer revenue slightly in the short term
but may significantly improve retention by aligning price to actual value
delivered — especially for Small customers with fewer active users.
''')

---

## 5. Revenue-to-User Efficiency

**Business Question:** Which plans are over-provisioned or underutilized?

We measure **seat utilization** (active users ÷ max users) and **revenue per active user** across all plans. Low utilization means customers are paying for unused capacity.

In [ ]:
# ── Utilization by plan ───────────────────────────────────────────────────
util = master.groupby('plan_name').agg(
    avg_active_users = ('active_users',   'mean'),
    max_users        = ('max_users',       'mean'),
    monthly_price    = ('monthly_price',   'mean')
).reset_index()

util['utilization_pct']      = (util['avg_active_users'] / util['max_users'] * 100).round(2)
util['unused_seats']         = (util['max_users'] - util['avg_active_users']).round(2)
util['rev_per_active_user']  = (util['monthly_price'] / util['avg_active_users']).round(2)
util['avg_active_users']     = util['avg_active_users'].round(2)

util_display = util[[
    'plan_name', 'monthly_price', 'max_users',
    'avg_active_users', 'unused_seats',
    'utilization_pct', 'rev_per_active_user'
]].copy()
util_display.columns = [
    'Plan', 'List Price ($)', 'Max Users',
    'Avg Active Users', 'Avg Unused Seats',
    'Utilization (%)', 'Rev / Active User ($)'
]
util_display = util_display.sort_values('Utilization (%)')

print('Revenue-to-User Efficiency by Plan')
print('=' * 80)
print(util_display.to_string(index=False))

In [ ]:
print('Business Interpretation')
print('=' * 60)
print('''
Utilization tells us what fraction of the seat capacity customers actually use:

  Basic      :  23.6% utilization — customers use only 1.2 of 5 allowed seats
  Pro        :  37.8% utilization — customers use about 7.6 of 20 seats
  Business   :  54.4% utilization — customers use ~27 of 50 seats
  Enterprise :  63.2% utilization — customers use ~126 of 200 seats

The Basic plan has the lowest utilization (23.6%), meaning customers are only
using 1 in 5 available seats on average. However, since Basic is the cheapest
plan, the financial risk per customer is low.

The Pro plan is more concerning: customers use fewer than 8 of 20 seats (37.8%),
yet pay $79/month. This unused capacity is a common churn trigger — customers
perceive low value when they are paying for seats they never fill.

Revenue per active user drops dramatically as plans scale up:
  Basic: $24.62/user → Pro: $10.46/user → Business: $5.48/user → Enterprise: $2.37/user

This confirms that higher plans offer significant per-user price efficiency,
which is a strong upsell argument for customers approaching their seat limits.
''')

---

## 6. Retention Opportunity Analysis

**Business Question:** Which segments create the biggest retention risk?

We analyze churn rates across key at-risk segments identified in prior analyses: EdTech customers, Small customers on higher plans, Large customers on Pro, and low-usage customers.

In [ ]:
# ── Build customer-level segment table ────────────────────────────────────
churned_ids  = set(churn['customer_id'])
seg          = customers.merge(plans, on='plan_id').copy()
seg['churned'] = seg['customer_id'].isin(churned_ids).astype(int)

# Avg monthly revenue per customer
cust_avg_rev = master.groupby('customer_id')['subscription_revenue'].mean().reset_index()
cust_avg_rev.columns = ['customer_id', 'avg_monthly_rev']
seg = seg.merge(cust_avg_rev, on='customer_id', how='left')

# Avg sessions per customer
cust_avg_sess = master.groupby('customer_id')['sessions'].mean().reset_index()
cust_avg_sess.columns = ['customer_id', 'avg_sessions']
seg = seg.merge(cust_avg_sess, on='customer_id', how='left')

# ── Define risk segments ──────────────────────────────────────────────────
# Segment 1: EdTech
edtech_seg   = seg[seg['industry'] == 'EdTech']

# Segment 2: Small on Pro (over-provisioned)
sm_pro_seg   = seg[(seg['company_size'] == 'Small') & (seg['plan_name'] == 'Pro')]

# Segment 3: Large on Pro (under-served — paying entry level, likely to churn or need upgrade)
lg_pro_seg   = seg[(seg['company_size'] == 'Large') & (seg['plan_name'] == 'Pro')]

# Segment 4: Low-usage customers (avg sessions < 5)
low_sess_ids = cust_avg_sess[cust_avg_sess['avg_sessions'] < 5]['customer_id']
low_use_seg  = seg[seg['customer_id'].isin(low_sess_ids)]

# Segment 5: Overall baseline
overall_seg  = seg.copy()

# ── Build summary table ───────────────────────────────────────────────────
def seg_stats(df, label):
    return {
        'Segment'              : label,
        'Customers'            : len(df),
        'Churned'              : int(df['churned'].sum()),
        'Churn Rate (%)'       : round(df['churned'].mean() * 100, 1),
        'Avg Monthly Rev ($)'  : round(df['avg_monthly_rev'].mean(), 2)
    }

rows = [
    seg_stats(overall_seg, 'ALL CUSTOMERS (Baseline)'),
    seg_stats(edtech_seg,  'EdTech'),
    seg_stats(sm_pro_seg,  'Small + Pro'),
    seg_stats(lg_pro_seg,  'Large + Pro'),
    seg_stats(low_use_seg, 'Low Usage (avg sessions < 5)')
]

retention_df = pd.DataFrame(rows)

print('Retention Risk by Segment')
print('=' * 78)
print(retention_df.to_string(index=False))

In [ ]:
# ── EdTech churn breakdown by plan ────────────────────────────────────────
edtech_plan = seg[seg['industry'] == 'EdTech'].groupby('plan_name').agg(
    Customers    = ('customer_id', 'count'),
    Churned      = ('churned',     'sum'),
).reset_index()
edtech_plan['Churn Rate (%)'] = (edtech_plan['Churned'] / edtech_plan['Customers'] * 100).round(1)
edtech_plan.columns = ['Plan', 'Customers', 'Churned', 'Churn Rate (%)']

print('EdTech Churn by Plan')
print('=' * 42)
print(edtech_plan.to_string(index=False))

In [ ]:
print('Business Interpretation')
print('=' * 60)
print('''
Retention risk is not evenly distributed across NovaSuite's customer base:

1. EdTech has the single highest churn rate at 15.0% — more than double
   the overall baseline of 7.2%. EdTech customers on the Business plan
   are the worst sub-segment at 23.5% churn.

2. Small + Pro customers churn at 11.0% — nearly double the 5.2% seen
   in Small + Basic customers. This confirms the plan mismatch hypothesis:
   Small customers are being upsold to Pro but not finding enough value.

3. Large + Pro is a surprising risk: 16.7% churn rate. Large companies
   on the entry-level Pro plan (20 seats max) likely outgrow the plan
   before they can be upgraded — leading to frustration and exit.

4. Low-usage customers (avg sessions < 5/month) represent 117 customers
   (23.4% of the base) but do not have dramatically higher churn in raw
   rate terms. However, low sessions is a leading indicator — these
   customers are at risk of future disengagement.

5. Notably, EdTech Enterprise customers (n=8) have 0% churn — suggesting
   the product does serve large EdTech institutions well, but is misfit
   for smaller EdTech companies on lower-tier plans.
''')

---

## 7. High-Value At-Risk Customers

**Business Question:** Which customers should be prioritized for retention?

We identify customers who combine **above-median revenue** with **below-median session activity** — customers who are paying well but not engaging, making them vulnerable to churn.

In [ ]:
# ── Customer-level aggregates ─────────────────────────────────────────────
cust_stats = master.groupby('customer_id').agg(
    avg_monthly_rev   = ('subscription_revenue', 'mean'),
    avg_sessions      = ('sessions',             'mean'),
    avg_feature_usage = ('feature_usage',        'mean'),
    avg_active_users  = ('active_users',         'mean')
).reset_index()

cust_stats['avg_monthly_rev']   = cust_stats['avg_monthly_rev'].round(2)
cust_stats['avg_sessions']      = cust_stats['avg_sessions'].round(2)
cust_stats['avg_feature_usage'] = cust_stats['avg_feature_usage'].round(2)
cust_stats['avg_active_users']  = cust_stats['avg_active_users'].round(2)

# Attach segment info
seg_info = seg[['customer_id', 'company_size', 'industry', 'plan_name', 'churned']].copy()
at_risk  = cust_stats.merge(seg_info, on='customer_id', how='left')

# ── Thresholds: median revenue (above) + median sessions (below) ──────────
median_rev  = at_risk['avg_monthly_rev'].median()
median_sess = at_risk['avg_sessions'].median()

print(f'Median avg monthly revenue : ${median_rev:.2f}')
print(f'Median avg sessions/month  : {median_sess:.2f}')
print()

In [ ]:
# ── High-value, low-engagement customers ─────────────────────────────────
hv_atrisk = at_risk[
    (at_risk['avg_monthly_rev'] >= median_rev) &
    (at_risk['avg_sessions']    <  median_sess)
].copy()

hv_atrisk = hv_atrisk.sort_values('avg_monthly_rev', ascending=False)

print(f'High-Value At-Risk Customers: {len(hv_atrisk)}')
print(f'Already churned within this group: {int(hv_atrisk["churned"].sum())}')
print(f'Churn rate within group: {round(hv_atrisk["churned"].mean()*100, 1)}%')
print()

# Display top 15
hv_display = hv_atrisk[[
    'customer_id', 'plan_name', 'industry', 'company_size',
    'avg_monthly_rev', 'avg_sessions', 'avg_feature_usage', 'churned'
]].head(15).copy()
hv_display.columns = [
    'Customer ID', 'Plan', 'Industry', 'Size',
    'Avg Rev/Month ($)', 'Avg Sessions', 'Avg Feature Usage', 'Churned'
]

print('Top 15 High-Value At-Risk Customers (by Revenue)')
print('=' * 90)
print(hv_display.to_string(index=False))

In [ ]:
# ── Segment breakdown within high-value at-risk group ────────────────────
hv_plan_dist = hv_atrisk.groupby('plan_name').agg(
    Customers           = ('customer_id',     'count'),
    Churned             = ('churned',          'sum'),
    Avg_Rev             = ('avg_monthly_rev',  'mean'),
    Avg_Sessions        = ('avg_sessions',     'mean')
).reset_index()
hv_plan_dist['Churn Rate (%)'] = (hv_plan_dist['Churned'] / hv_plan_dist['Customers'] * 100).round(1)
hv_plan_dist['Avg_Rev']        = hv_plan_dist['Avg_Rev'].round(2)
hv_plan_dist['Avg_Sessions']   = hv_plan_dist['Avg_Sessions'].round(2)
hv_plan_dist.columns = ['Plan', 'Customers', 'Churned', 'Avg Rev ($)', 'Avg Sessions', 'Churn Rate (%)']

print('High-Value At-Risk — Breakdown by Plan')
print('=' * 65)
print(hv_plan_dist.to_string(index=False))

In [ ]:
hv_total_rev_risk = round(hv_atrisk['avg_monthly_rev'].sum(), 2)

print('Business Interpretation')
print('=' * 60)
print(f'''
There are {len(hv_atrisk)} customers who generate above-median revenue
but have below-median session engagement.

Combined monthly revenue at risk: ${hv_total_rev_risk:,.2f}/month
Annualized revenue at risk      : ${hv_total_rev_risk * 12:,.2f}/year

Within this group, {int(hv_atrisk['churned'].sum())} customers have already churned
(churn rate: {round(hv_atrisk['churned'].mean()*100, 1)}% — slightly below overall 7.2%).

The Pro plan dominates this list — high-revenue Pro customers with low
engagement represent the highest-priority outreach targets. These customers
are paying close to full list price but not deriving visible value from usage.

Low feature_usage scores (many below 4.0) within this group confirm that
these customers are not deeply integrated into NovaSuite's feature set,
making them vulnerable to competitive displacement or budget cuts.

A proactive success check-in or onboarding refresh program targeted at
this cohort could protect a meaningful share of recurring revenue.
''')

---

## 8. Pricing & Retention Findings Summary

The following observations are drawn directly from the data. No recommendations are made in this section — these findings will feed the Strategy Recommendations section.

---

### 💰 Pricing Findings

1. **Basic plan discount leakage is 9.9%.** The average realized revenue of \$26.66 falls \$2.34 short of the \$29 list price every month, per row — not a rounding error but a structural policy gap.

2. **Pro plan drives 27.7% of total subscription revenue** across 177 customers, making it the single most financially significant plan. Its 10.75% discount leakage rate is the second-highest among plans.

3. **Enterprise plan has the worst discount leakage at 19.57%** — nearly 1-in-5 dollars of Enterprise base revenue is discounted away. The average Enterprise discount is \$58.51/month per row.

4. **Business plan loses 16.39% of base revenue to discounts.** At \$24.42 average discount per row, Business plan customers are receiving the second-largest absolute discount of any plan.

5. **EdTech is the most discounted industry at 22.91% leakage** — the highest of all 8 industries, and nearly double FinTech's 12.07%. This is particularly notable given that EdTech also has the highest churn rate.

6. **Retail (17.56%) and Manufacturing (15.64%) are the 2nd and 3rd most discounted industries** — suggesting broad discount overuse outside the core FinTech and Healthcare verticals.

7. **Annualized discount leakage across all plans is approximately \$47,400/year** (based on \$3,950/month average across 39 months of data).

8. **Basic plan seat utilization is only 23.6%** — customers use 1.2 out of 5 allowed seats. Pro plan utilization is 37.8% (7.6 of 20 seats). No plan exceeds 64% utilization on average.

9. **Revenue per active user collapses at higher plan tiers:** Basic \$24.62 → Pro \$10.46 → Business \$5.48 → Enterprise \$2.37. Higher plans offer more seats per dollar, but few customers fill those seats.

10. **Small customers on Pro average only 5.46 active users** against a 20-seat ceiling. Under a usage-based model (\$20 + \$8/user), they would pay \$63.67 vs \$71.37 currently — an 11% reduction that could reduce churn while accepting a short-term revenue dip.

---

### 🔄 Retention Findings

1. **EdTech has a 15.0% churn rate** — more than double the overall 7.2% baseline. EdTech is the only industry that clearly diverges from the norm in churn behavior.

2. **EdTech on Business plan churns at 23.5%** — the highest single segment-plan combination observed. In contrast, EdTech on Enterprise (n=8) has 0% churn, suggesting a product fit issue at lower tiers.

3. **Small + Pro customers churn at 11.0%** vs 5.2% for Small + Basic customers. The Pro plan appears to be a poor fit for Small companies — they churn at twice the rate of Small customers on a cheaper plan.

4. **Large + Pro customers churn at 16.7%** — the highest churn rate of any size × plan combination analyzed. Large companies on a plan capped at 20 users are likely outgrowing their plan before an upgrade conversation occurs.

5. **117 customers (23.4% of total) have average sessions below 5/month.** While their raw churn rate is not dramatically higher today, consistent low engagement is a documented precursor to churn (as established in the Churn Analysis section).

6. **27 customers generate above-median revenue but have below-median session activity**, representing a concentrated at-risk revenue pool of \$2,498/month (\$29,980/year annualized).

7. **Expansion revenue is concentrated and infrequent.** Enterprise customers generate expansion revenue in only 20.7% of months; Business in 16.8%; Pro in 6.2%; Basic in just 2.0%. The vast majority of months across all plans generate zero expansion revenue.

8. **EdTech accounts for 13.4% of total subscription revenue** (\$119,293 all-time) despite having the highest churn rate. This segment represents both a significant revenue concentration risk and a potential turnaround opportunity.

9. **Medium + Business customers churn at 9.8%** — higher than both Medium + Basic (2.6%) and Medium + Pro (8.1%). The Business plan may be over-sold to Medium customers who don't need its full feature set.

10. **The overall 7.2% all-time churn rate is unevenly distributed:** the bottom half of segments (EdTech, Small+Pro, Large+Pro) have churn rates ranging from 11–23.5%, while stable segments (Medium+Basic, Large+Business) maintain rates below 5%.